In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import torch
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel, AutoConfig
import joblib
import openai
import shutil
import zipfile
from openai import OpenAI

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

def spacy_sent_tokenize(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents]

In [ ]:
models = ["secbert", "cybertuned", "ctibert"]

for model in models:
    zip_path = f"/content/drive/My Drive/mitre_model_{model}.zip"
    local_path = f"/content/mitre_model_{model}.zip"
    extract_path = f"/content/mitre_model_{model}"

    shutil.copy(zip_path, local_path)
    with zipfile.ZipFile(local_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)


In [ ]:
# import zipfile

# with zipfile.ZipFile("mitre_model_ctibert.zip", 'r') as zip_ref:
#     zip_ref.extractall("mitre_model_ctibert")

# with zipfile.ZipFile("mitre_model_cybertuned.zip", 'r') as zip_ref:
#     zip_ref.extractall("mitre_model_cybertuned")

# with zipfile.ZipFile("mitre_model_secbert.zip", 'r') as zip_ref:
#     zip_ref.extractall("mitre_model_secbert")

In [ ]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

MAX_LEN = 512
VALID_BATCH_SIZE = 32
test_params = {'batch_size': VALID_BATCH_SIZE, 'shuffle': False, 'num_workers': 0}

In [ ]:
class Triage(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, index):
        sentence = str(self.df.sentence[index])
        inputs = self.tokenizer.encode_plus(
            sentence,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=False
        )
        ids  = torch.tensor(inputs['input_ids'], dtype=torch.long)
        mask = torch.tensor(inputs['attention_mask'], dtype=torch.long)
        return {
            'input_ids': ids,        # used by CSV eval loops
            'attention_mask': mask,  # used by CSV eval loops
            'ids': ids,              # used by CTI/ensemble helpers
            'mask': mask,            # used by CTI/ensemble helpers
            'sentence': sentence
        }

    def __len__(self):
        return len(self.df)


In [ ]:
class SecBertClassifier(torch.nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name, output_hidden_states=True)
        self.model = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = torch.nn.Dropout(0.3)
        self.pre_classifier = torch.nn.Linear(config.hidden_size, config.hidden_size)
        self.classifier = torch.nn.Linear(config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        mean_pooled = torch.sum(hidden * attention_mask.unsqueeze(-1), dim=1) / attention_mask.sum(1, keepdim=True)
        x = self.pre_classifier(mean_pooled)
        x = torch.nn.ReLU()(x)
        x = self.dropout(x)
        return self.classifier(x)

# Loading assets
encoder = joblib.load("mitre_model_secbert/label_encoder.pkl")
LABELS = len(encoder.classes_)
tokenizer = AutoTokenizer.from_pretrained("jackaduma/SecBERT")

# Loading dataset
df = pd.read_csv("dataset_mixed.csv")
dataset = Triage(df, tokenizer)
loader = DataLoader(dataset, batch_size=8)

# Loading model
model = SecBertClassifier("jackaduma/SecBERT", LABELS)
model.load_state_dict(torch.load("mitre_model_secbert/model.pt", map_location="cpu"))
model.eval()

# Inference
probs = []
sentences = []
with torch.no_grad():
    for batch in loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        outputs = model(input_ids, attention_mask)
        softmax_probs = F.softmax(outputs, dim=1)
        probs.extend(softmax_probs.tolist())
        sentences.extend(batch['sentence'])

# Saving sentences
pd.DataFrame(probs).to_csv("secbert_probs.csv", index=False)
pd.DataFrame({"sentence": sentences}).to_csv("sentences.csv", index=False)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

predictions = np.argmax(probs, axis=1)
df_ground_truth = pd.read_csv("dataset_mixed.csv")
true_labels = encoder.transform(df_ground_truth['label_tec'])
print("Accuracy of Secbert on labeled dataset:")
print(f"Accuracy: {accuracy_score(true_labels, predictions) * 100:.2f}%")

In [ ]:
class CTIBERTClassifier(torch.nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name, output_hidden_states=True)
        self.model = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(config.hidden_size, config.hidden_size // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(config.hidden_size // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        pooled = torch.sum(hidden * attention_mask.unsqueeze(-1), dim=1) / attention_mask.sum(1, keepdim=True)
        return self.classifier(self.dropout(pooled))

# Loading assets
encoder = joblib.load("mitre_model_ctibert/label_encoder.pkl")
LABELS = len(encoder.classes_)
tokenizer = AutoTokenizer.from_pretrained("ibm-research/CTI-BERT")

# Loading dataset
df = pd.read_csv("dataset_mixed.csv")
dataset = Triage(df, tokenizer)
loader = DataLoader(dataset, batch_size=8)

# Loading model
model = CTIBERTClassifier("ibm-research/CTI-BERT", LABELS)
model.load_state_dict(torch.load("mitre_model_ctibert/ctibert_model.pt", map_location="cpu"))
model.eval()

# Inference
probs = []
sentences = []
with torch.no_grad():
    for batch in loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        outputs = model(input_ids, attention_mask)
        softmax_probs = F.softmax(outputs, dim=1)
        probs.extend(softmax_probs.tolist())
        sentences.extend(batch['sentence'])

# Saving sentences
pd.DataFrame(probs).to_csv("ctibert_probs.csv", index=False)
pd.DataFrame({"sentence": sentences}).to_csv("sentences.csv", index=False)

In [ ]:
predictions = np.argmax(probs, axis=1)
df_ground_truth = pd.read_csv("dataset_mixed.csv")
true_labels = encoder.transform(df_ground_truth['label_tec'])
print("Accuracy of CTI-BERT on labeled dataset:")
print(f"Accuracy: {accuracy_score(true_labels, predictions) * 100:.2f}%")

In [ ]:
class CyberTunedClassifier(torch.nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name, output_hidden_states=True)
        self.model = AutoModel.from_pretrained(model_name, config=config)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(config.hidden_size, config.hidden_size // 2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(config.hidden_size // 2, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        mean_pooled = torch.sum(hidden * attention_mask.unsqueeze(-1), dim=1) / attention_mask.sum(1, keepdim=True)
        return self.classifier(self.dropout(mean_pooled))

# Loading assets
encoder = joblib.load("mitre_model_cybertuned/label_encoder.pkl")
LABELS = len(encoder.classes_)
tokenizer = AutoTokenizer.from_pretrained("s2w-ai/CyBERTuned-SecurityLLM")

# Loading dataset
df = pd.read_csv("dataset_mixed.csv")
dataset = Triage(df, tokenizer)
loader = DataLoader(dataset, batch_size=8)

# Loading model
model = CyberTunedClassifier("s2w-ai/CyBERTuned-SecurityLLM", LABELS)
model.load_state_dict(torch.load("mitre_model_cybertuned/cybertuned_model.pt", map_location="cpu"))
model.eval()

# Inference
probs = []
sentences = []
with torch.no_grad():
    for batch in loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        outputs = model(input_ids, attention_mask)
        softmax_probs = F.softmax(outputs, dim=1)
        probs.extend(softmax_probs.tolist())
        sentences.extend(batch['sentence'])

# Saving sentences
pd.DataFrame(probs).to_csv("cybertuned_probs.csv", index=False)
pd.DataFrame({"sentence": sentences}).to_csv("sentences.csv", index=False)

In [ ]:
predictions = np.argmax(probs, axis=1)
df_ground_truth = pd.read_csv("dataset_mixed.csv")
true_labels = encoder.transform(df_ground_truth['label_tec'])
print("Accuracy of Cybertuned on labeled dataset:")
print(f"Accuracy: {accuracy_score(true_labels, predictions) * 100:.2f}%")

In [ ]:
import pandas as pd
import joblib
import numpy as np

ctibert_probs = pd.read_csv("ctibert_probs.csv").values
cybertuned_probs = pd.read_csv("cybertuned_probs.csv").values
secbert_probs = pd.read_csv("secbert_probs.csv").values
sentences = pd.read_csv("sentences.csv")["sentence"].tolist()


# Soft voting
avg_probs = (cybertuned_probs + secbert_probs + ctibert_probs)/3

top_k = 3
topk_indices = np.argsort(avg_probs, axis=1)[:, -top_k:][:, ::-1]
topk_labels = [encoder.inverse_transform(row) for row in topk_indices]

predictions = np.argmax(avg_probs, axis=1)

# Decoding
encoder = joblib.load("mitre_model_secbert/label_encoder.pkl")
labels = encoder.inverse_transform(predictions)

# Saving results
pd.DataFrame({
    "sentence": sentences,
    "predicted_label": labels,
    "confidence": avg_probs.max(axis=1)
}).to_csv("ensemble_soft_voting_results.csv", index=False)
print("Saved ensemble_soft_voting_results.csv")

In [ ]:
df_ground_truth = pd.read_csv("dataset_mixed.csv")
true_labels = encoder.transform(df_ground_truth['label_tec'])

# Ensuring lengths match
assert len(true_labels) == len(labels), "Mismatch between prediction and ground truth!"

from sklearn.metrics import accuracy_score, classification_report

print("Accuracy on labeled dataset:")
print(f"Accuracy: {accuracy_score(true_labels, predictions) * 100:.2f}%")


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Ensemble (soft voting) metrics on dataset_mixed.csv
macro_p = precision_score(true_labels, predictions, average="macro", zero_division=0)
macro_r = recall_score(true_labels, predictions,   average="macro", zero_division=0)
macro_f = f1_score(true_labels, predictions,       average="macro", zero_division=0)
acc     = accuracy_score(true_labels, predictions)

print("Ensemble (soft voting) performance on labeled dataset:")
print(f"Accuracy:          {acc*100:.2f}%")
print(f"Precision (macro): {macro_p:.4f}")
print(f"Recall (macro):    {macro_r:.4f}")
print(f"F1 (macro):        {macro_f:.4f}")




In [ ]:
import numpy as np

# probs from each model: shape [n_samples, n_classes]
P_sec = secbert_probs
P_cyb = cybertuned_probs
P_cti = ctibert_probs

y_true = true_labels

def grid_weights(step=0.05):
    """Generate all (w1,w2,w3) >= 0, sum=1, in increments of 'step'."""
    S = int(round(1/step))
    for a in range(S+1):
        for b in range(S+1-a):
            c = S - a - b
            yield (a/S, b/S, c/S)

def evaluate_combo(w, metric="accuracy"):
    w1, w2, w3 = w
    avg = w1*P_sec + w2*P_cyb + w3*P_cti
    pred = avg.argmax(axis=1)
    if metric == "accuracy":
        return (pred == y_true).mean()
    # from sklearn.metrics import f1_score
    # return f1_score(y_true, pred, average="macro")
    return (pred == y_true).mean()

best = (-1, None)  # (score, weights)
top = []           # keeping top-K
step = 0.05        # trying 0.10 for speed, 0.02 or 0.01 for finer search

for w in grid_weights(step):
    score = evaluate_combo(w, metric="accuracy")
    top.append((score, w))
    if score > best[0]:
        best = (score, w)

top.sort(reverse=True, key=lambda x: x[0])
print("Best accuracy:", round(best[0], 4), "at weights (SecBERT, CyBERTuned, CTI-BERT):", best[1])
for s,(w1,w2,w3) in top[:10]:
    print(f"{s:.4f} -> ({w1:.2f}, {w2:.2f}, {w3:.2f})")


In [ ]:
from nltk.tokenize import sent_tokenize

def remove_empty_lines(text):
	lines = text.split("\n")
	non_empty_lines = [line for line in lines if line.strip() != ""]

	string_without_empty_lines = ""
	for line in non_empty_lines:
		if line != "\n":
			string_without_empty_lines += line + "\n"

	return string_without_empty_lines

def combine_text(list_of_text):
    combined_text = ' '.join(list_of_text)
    return combined_text


In [ ]:
!python -m nltk.downloader punkt

In [ ]:
import yaml
import re

def repl(matchobj):
    return ","+ matchobj.group(1) + ","

def load_regex(filename):
    regex_list = []
    with open(filename, 'r') as val:
        document = yaml.safe_load(val)
        regex_list = document
    return regex_list

def apply_regex_to_string(regex_list, text):
    new_text = text
    for rex in regex_list:
        pat = r'{}'.format(rex.get('regex', '').strip())
        repl = rex.get('code', '') + " "
        new_text = re.sub(pat, repl, new_text)
    return new_text



In [ ]:
fin6_tec_2 = ['T1134', 'T1059', 'T1562', 'T1036', 'T1588', 'T1003', 'T1021', 'T1569', 'T1078', 'T1102',
        'T1087', 'T1482', 'T1069', 'T1018', 'T1016', 'T1548', 'T1071', 'T1185', 'T1059',
        'T1543', 'T1132', 'T1005', 'T1001', 'T1140', 'T1573', 'T1068', 'T1083',
        'T1564', 'T1562', 'T1070', 'T1105', 'T1056', 'T1112', 'T1046', 'T1095', 'T1027', 'T1137',
        'T1003', 'T1069', 'T1057', 'T1055', 'T1572', 'T1572', 'T1090', 'T1012', 'T1620', 'T1021',
        'T1018', 'T1029', 'T1113', 'T1518', 'T1553', 'T1218', 'T1049', 'T1007', 'T1569', 'T1550',
        'T1078', 'T1047']

fin6_tec_1 = ['T1087', 'T1560', 'T1119', 'T1547', 'T1110', 'T1059', 'T1074', 'T1573', 'T1068', 'T1070',
        'T1046', 'T1003', 'T1572', 'T1021', 'T1018', 'T1053', 'T1078', 'T1003']

# MenuPass [8]

menuPass_tec_8 = ['T1560', 'T1119', 'T1059', 'T1005', 'T1074', 'T1210', 'T1083', 'T1574', 'T1106', 'T1027', 'T1003', 'T1199', 'T1078', 'T1047']

adFind = ['T1087', 'T1482', 'T1069', 'T1018', 'T1016']

certutil = ['T1140', 'T1105', 'T1553']

quasarRAT = ['T1059', 'T1555', 'T1573', 'T1105', 'T1056', 'T1112', 'T1090', 'T1021', 'T1053', 'T1553', 'T1082', 'T1552', 'T1125']

menuPass_tec_8.extend(adFind)
menuPass_tec_8.extend(certutil)
menuPass_tec_8.extend(quasarRAT)

# MenuPass [2]

menuPass_tec_2 = ['T1583', 'T1560', 'T1568', 'T1070', 'T1056', 'T1036', 'T1105', 'T1566', 'T1021', 'T1199', 'T1204', 'T1078']

poisonIvy = ['T1010', 'T1547', 'T1059', 'T1543', 'T1005', 'T1074', 'T1573', 'T1105', 'T1056', 'T1112', 'T1027',
'T1055', 'T1014']

menuPass_tec_2.extend(poisonIvy)

# WizardSpider [2]

wizardSpider_tec_2 = ['T1547', 'T1059', 'T1562', 'T1135', 'T1566', 'T1055', 'T1021', 'T1053', 'T1558', 'T1204', 'T1047']

bloodHound = ['T1087', 'T1560', 'T1059', 'T1482', 'T1615', 'T1106', 'T1201', 'T1069', 'T1018', 'T1033']

cobaltStrike = ['T1548', 'T1134', 'T1087', 'T1071', 'T1197', 'T1185', 'T1059', 'T1043', 'T1543', 'T1132', 'T1005', 'T1001', 'T1030', 'T1140', 'T1573', 'T1203', 'T1068', 'T1083', 'T1564', 'T1562', 'T1070', 'T1105', 'T1056', 'T1112', 'T1026', 'T1106', 'T1046', 'T1135', 'T1095', 'T1027', 'T1137', 'T1003', 'T1069', 'T1057', 'T1055', 'T1572', 'T1090', 'T1012', 'T1620', 'T1021', 'T1018', 'T1029', 'T1113', 'T1518', 'T1553', 'T1218', 'T1016', 'T1049', 'T1007', 'T1569', 'T1550', 'T1078', 'T1047']

empire =  ['T1548', 'T1134', 'T1087', 'T1557', 'T1071', 'T1560', 'T1547', 'T1217', 'T1115', 'T1059', 'T1043', 'T1136', 'T1543', 'T1555', 'T1484', 'T1482', 'T1114', 'T1573', 'T1546', 'T1068', 'T1083', 'T1574', 'T1210', 'T1615', 'T1567', 'T1070',  'T1056', 'T1105', 'T1056', 'T1106', 'T1046', 'T1135', 'T1040', 'T1027', 'T1003', 'T1057', 'T1055', 'T1021', 'T1053', 'T1113', 'T1518', 'T1558', 'T1082', 'T1016', 'T1049', 'T1569', 'T1127', 'T1552', 'T1550', 'T1125', 'T1102', 'T1047']

mimikatz = ['T1134', 'T1098', 'T1547', 'T1555', 'T1003', 'T1207', 'T1558', 'T1552', 'T1550']

ping = ['T1018']

ryuk = ['T1134', 'T1547', 'T1059', 'T1486', 'T1083', 'T1222', 'T1562', 'T1490', 'T0828', 'T1036', 'T1106', 'T1027', 'T1057', 'T1055', 'T1021', 'T1053', 'T1489', 'T1082', 'T1614', 'T1016', 'T1205', 'T1078']

trickBot = ['T1087', 'T1087', 'T1071', 'T1547', 'T1185', 'T1110', 'T1059', 'T1059', 'T1043', 'T1543', 'T1555', 'T1555', 'T1132', 'T1005', 'T1140', 'T1482', 'T1573', 'T1041', 'T1210', 'T1008', 'T1083', 'T1495', 'T1562', 'T1105', 'T1056', 'T1559', 'T1036', 'T1112', 'T1106', 'T1135', 'T1571', 'T1027', 'T1027', 'T1069', 'T1566', 'T1566', 'T1542', 'T1057', 'T1055', 'T1055', 'T1090', 'T1219', 'T1021', 'T1018', 'T1053', 'T1553', 'T1082', 'T1016', 'T1033', 'T1007', 'T1552', 'T1552', 'T1204', 'T1497']

wizardSpider_tec_2.extend(bloodHound)
wizardSpider_tec_2.extend(cobaltStrike)
wizardSpider_tec_2.extend(empire)
wizardSpider_tec_2.extend(mimikatz)
wizardSpider_tec_2.extend(ping)
wizardSpider_tec_2.extend(ryuk)
wizardSpider_tec_2.extend(trickBot)

#WizardSpider [7]

wizardSpider_tec_7 = ['T1087', 'T1059', 'T1048', 'T1210', 'T1562', 'T1027', 'T1021', 'T1018', 'T1489', 'T1518', 'T1558', 'T1082', 'T1569']

adFind = ['T1087', 'T1482', 't1069', 'T1018', 'T1016']

#CobaltStrike

net = ['T1087', 'T1087', 'T1136', 'T1136', 'T1070', 'T1135', 'T1201', 'T1069', 'T1069', 'T1021', 'T1018', 'T1049', 'T1007', 'T1569', 'T1124']

nltest = ['T1482', 'T1018', 'T1016']

#Ping

#Ryuk

wizardSpider_tec_7.extend(adFind)
wizardSpider_tec_7.extend(cobaltStrike)
wizardSpider_tec_7.extend(net)
wizardSpider_tec_7.extend(nltest)
wizardSpider_tec_7.extend(ping)
wizardSpider_tec_7.extend(ryuk)

In [ ]:
fin6_files = ['./Follow The Money-Dissecting the Operations of the Cyber Crime Group FIN6[1].txt',
                './Pick-Six-Intercepting a FIN6 Intrusion, an Actor Recently Tied to Ryuk and LockerGoga Ransomware[2].txt',
                './intelligence_summary.txt']

menuPass_files = ['./2018_12_20_united_states_v_zhu_hua_indictment[2].txt',
                './Japan-Linked Organizations Targeted in Long-Running and Sophisticated Attack Campaign[8].txt']

wizardSpider_files = ['./Ryuk’s Return[7].txt',
                     './Ransomware Activity Targeting the Healthcare and Public Health Sector. Retrieved October 28, 2020[2].txt']

In [ ]:
file_name = wizardSpider_files[1]
techniques = wizardSpider_tec_2

In [ ]:
# file_name = wizardSpider_files[0]
# techniques = wizardSpider_tec_7

In [ ]:
#Read report text from txt file
# import nltk
# nltk.download('punkt', force=True)  # Force fresh download

# from nltk.tokenize import sent_tokenize

lines = []
file_paths = [file_name]
for file_path in file_paths:
    with open(file_path) as f:
        lines += f.readlines()
import re
## Apply regex
regex_list = load_regex("regex.yml")

text = combine_text(lines)
text = re.sub('(%(\w+)%(\/[^\s]+))', repl, text)
text = apply_regex_to_string(regex_list, text)
text = re.sub('\(.*?\)', '', text)
text = remove_empty_lines(text)
text = text.strip()
sentences = spacy_sent_tokenize(text)
# double_sentences = []

# for i in range(1, len(sentences)):
#     new_sen = sentences[i-1] + sentences[i]
#     double_sentences.append(new_sen)

# data = {'sentence': sentences}
# df = pd.DataFrame(data, columns=['sentence'])
# sentence_set = Triage(df, tokenizer, MAX_LEN)
# testing_loader = DataLoader(sentence_set, **test_params)

In [ ]:
# len(sentences)

In [ ]:
# predicted = []
# predict_proba_scores = []
# with torch.no_grad():
#       for i, data in enumerate(testing_loader, 0):
#           x = data['ids'].to(device, dtype = torch.long)
#           mask = data['mask'].to(device, dtype = torch.long)

#           scores = model(x, mask)
#           _, predictions = scores.max(1)
#           proba_scores = torch.nn.functional.softmax(scores, dim=1)

#           predicted += predictions
#           predict_proba_scores += proba_scores


In [ ]:
# predicted = [ pred.item() for pred in predicted]


In [ ]:
# predict_proba_scores = [pred.max().item() for pred in predict_proba_scores]


In [ ]:
# print("Max predicted index:", max(predicted))
# print("Encoder classes available:", len(encoder.classes_))


In [ ]:
encoder = joblib.load("mitre_model_secbert/label_encoder.pkl")
LABELS = len(encoder.classes_)


tok_sec  = AutoTokenizer.from_pretrained("jackaduma/SecBERT")
tok_cyb  = AutoTokenizer.from_pretrained("s2w-ai/CyBERTuned-SecurityLLM")
tok_cti  = AutoTokenizer.from_pretrained("ibm-research/CTI-BERT")

secbert_model = SecBertClassifier("jackaduma/SecBERT", LABELS)
secbert_model.load_state_dict(torch.load("mitre_model_secbert/model.pt", map_location="cpu"))
secbert_model.eval()

cybertuned_model = CyberTunedClassifier("s2w-ai/CyBERTuned-SecurityLLM", LABELS)
cybertuned_model.load_state_dict(torch.load("mitre_model_cybertuned/cybertuned_model.pt", map_location="cpu"))
cybertuned_model.eval()

ctibert_model = CTIBERTClassifier("ibm-research/CTI-BERT", LABELS)
ctibert_model.load_state_dict(torch.load("mitre_model_ctibert/ctibert_model.pt", map_location="cpu"))
ctibert_model.eval()



In [ ]:
# Ensemble Helper
import numpy as np
import torch.nn.functional as F

def probs_for_sentences(model, tokenizer, sentences, batch_size=32):
    df_tmp = pd.DataFrame({"sentence": sentences})
    ds_tmp = Triage(df_tmp, tokenizer, MAX_LEN)
    dl_tmp = DataLoader(ds_tmp, batch_size=batch_size, shuffle=False, num_workers=0)
    outs = []
    model.eval()
    with torch.no_grad():
        for b in dl_tmp:
            ids = b['input_ids'] if 'input_ids' in b else b['ids']
            msk = b['attention_mask'] if 'attention_mask' in b else b['mask']
            logits = model(ids, msk)
            outs.append(F.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(outs) if outs else np.zeros((0, LABELS), dtype=np.float32)

p_sec = probs_for_sentences(secbert_model,   tok_sec, sentences)
p_cyb = probs_for_sentences(cybertuned_model, tok_cyb, sentences)
p_cti = probs_for_sentences(ctibert_model,    tok_cti, sentences)

avg_probs = (p_sec + p_cyb + p_cti) / 3.0
pred_idx  = avg_probs.argmax(axis=1)
pred_conf = avg_probs.max(axis=1)

predicted = encoder.inverse_transform(pred_idx)
predict_proba_scores = pred_conf.tolist()

print("Max predicted index:", int(pred_idx.max()))
print("Encoder classes available:", len(encoder.classes_))


In [ ]:
# predicted = encoder.inverse_transform(predicted)

In [ ]:
total = len(sentences)
correct = sum([1 for pred in predicted if pred in techniques])
accuracy = correct / total * 100
print(f"Raw Accuracy on CTI Report: {accuracy:.2f}%") # not true accuracy just an overlap check of known techniques and predicted techniques.

In [ ]:
def f_measure(recall, precision):
    if recall != 0 and precision != 0:
        return (2*precision*recall)/(precision+recall)
    else:
        return 0.01

In [ ]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
precisions = []
recalls = []
corrected_pred = []
accepted_pred = []
correct_on_uniques = []
f1s = []

print(len(predicted))
print('list of predicted techniques: ',predicted)

lines = len(predicted)

In [ ]:
for threshold in thresholds:
    # normalizing known techniques and accepted predictions
    tecs = {t.upper().strip() for t in techniques if isinstance(t, str)}
    accepted = [
        predicted[i].upper().strip()
        for i, p in enumerate(predict_proba_scores)
        if p > threshold
    ]
    acc_set = set(accepted)

    # confusion components on UNIQUE accepted labels
    TP = len(acc_set & tecs)
    FP = len(acc_set - tecs)
    FN = len(tecs - acc_set)

    precision = TP / (TP + FP) if (TP + FP) else 0.0
    recall    = TP / (TP + FN) if (TP + FN) else 0.0
    f1        = f_measure(recall=recall, precision=precision) if (precision and recall) else 0.0

    # store FRACTIONS (format as % only when printing)
    precisions.append(round(precision, 4))
    recalls.append(round(recall, 4))
    f1s.append(round(f1, 4))

    # extra Sanity
    corrected_pred.append(TP)                       # TP count
    accepted_pred.append(len(accepted))             # accepted with duplicates
    correct_on_uniques.append(f"{TP}/{len(acc_set)}")

    print(
        f"Threshold {threshold:.1f}: "
        f"F1={f1:.3f}, Precision={precision*100:.1f}%, Recall={recall*100:.1f}%"
    )


In [ ]:
import matplotlib.pyplot as plt

# Converting string-form recall values to fractions
recalls_numeric = []
for r in recalls:
    if isinstance(r, str) and "/" in r:
        num, den = map(int, r.split("/"))
        recalls_numeric.append(num / den)
    else:
        recalls_numeric.append(r)

# Converting precision % to fraction (0-1 scale)
precisions_fraction = [p / 100 for p in precisions]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions_fraction, marker='o', label='Precision')
plt.plot(thresholds, recalls_numeric, marker='s', label='Recall')
plt.plot(thresholds, f1s, marker='^', label='F1 Score')

plt.xlabel('Confidence Threshold')
plt.ylabel('Score (0–1)')
plt.title('Precision, Recall, and F1 Score vs Threshold')
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
class Classifier_results:
    def __init__(self, title, lines, accepted_preds, correct_preds, precisions, recalls, correct_uniques, f1s):
        self.title = title
        self.lines = lines
        self.accepted_preds = accepted_preds
        self.correct_preds = correct_preds
        self.precisions = precisions
        self.recalls = recalls
        self.correct_uniques = correct_uniques
        self.f1s = f1s

class CSVOutput:
    def __init__(self, document_title, classifiers):
        self.classifiers = classifiers
        self.document_title = document_title

    def printify_array(self, array, sep = ';'):
        return sep + sep.join(str(x) for x in array)

    def _save_classifier_outputs(self, f):
        for classifier in self.classifiers:
            f.write(classifier.title + '\n')
            f.write(str(classifier.lines) + ' sentences\n')
            f.write('Accepted Predictions: {}\n'.format(self.printify_array(classifier.accepted_preds)))
            f.write('Corrected Predictions: {}\n'.format(self.printify_array(classifier.correct_preds)))
            f.write('Precision%: {}\n'.format(self.printify_array(classifier.precisions)))
            f.write('Recall%: {}\n'.format(self.printify_array(classifier.recalls)))
            f.write('Correct predictions on uniques: {}\n\n'.format(self.printify_array(classifier.correct_uniques)))

    def _save_classifier_f1(self, path):
        with open(path+'/'+self.document_title+'_f1.txt', 'w') as f:
            for classifier in self.classifiers:
                f.write(classifier.title)
                f.write(self.printify_array(classifier.f1s)+ '\n')

    def write_to_file(self, path):
        self._save_classifier_f1(path)
        with open(path+'/'+self.document_title+'.csv', 'w') as f:
            f.write('Thresholds; 0,1; 0.2; 0.3; 0.4; 0.5; 0.6; 0.7; 0.8;\n')
            self._save_classifier_outputs(f)

    def append_to_file(self, path):
        self._save_classifier_f1(path)
        with open(path+'/'+self.document_title+'.csv', 'a') as f:
            self._save_classifier_outputs(f)

In [ ]:
import pandas as pd

print("Sentence count:", len(sentences))
print("Predictions count:", len(predicted))
print("Confidence scores count:", len(predict_proba_scores))

assert len(sentences) == len(predicted) == len(predict_proba_scores), "Mismatch between predictions and sentences!"

output_df = pd.DataFrame({
    'Sentence': sentences,
    'Predicted Technique': predicted,
    'Confidence Score': predict_proba_scores
})

output_df.to_csv("cti_report_predictions.csv", index=False)
print("Predictions saved to cti_report_predictions.csv")

In [ ]:
result = Classifier_results( title='Cti-Bert',
                              lines=lines,
                              accepted_preds=accepted_pred,
                              correct_preds=corrected_pred,
                              precisions=precisions,
                              recalls=recalls,
                              correct_uniques=correct_on_uniques,
                              f1s=f1s)

In [ ]:
fin6_ref_1_output = CSVOutput('ensemble_final_result', [result])
fin6_ref_1_output.write_to_file('.')

In [ ]:
import torch.nn.functional as F
import numpy as np

def model_probs(model, tokenizer, sentences):
    df_sen = pd.DataFrame({'sentence': sentences})
    ds = Triage(df_sen, tokenizer, MAX_LEN)
    dl = DataLoader(ds, **test_params)
    out = []
    with torch.no_grad():
        for batch in dl:
            x = batch['ids'].to(device)
            m = batch['mask'].to(device)
            logits = model(x, m)
            out.append(F.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(out) if out else np.zeros((0, LABELS), dtype=np.float32)

# def ensemble_predict(sentences, weights=(1/3, 1/3, 1/3)):
#     p_sec = model_probs(secbert,    tok_secbert,    sentences)
#     p_cyb = model_probs(cybertuned, tok_cybertuned, sentences)
#     p_cti = model_probs(ctibert,    tok_ctibert,    sentences)

#     # Handle empty input cleanly
#     if p_sec.shape[0] == 0:
#         return [], [], np.array([])

#     w1, w2, w3 = weights
#     avg = w1*p_sec + w2*p_cyb + w3*p_cti

#     idx = avg.argmax(axis=1)
#     labels = encoder.inverse_transform(idx)
#     conf = avg.max(axis=1)
#     return labels, idx, conf, avg


In [ ]:

import pandas as pd

# Normalization check
techniques_set = {t.upper().strip() for t in techniques if isinstance(t, str)}
pred_upper = [ (p.upper().strip() if isinstance(p, str) else str(p)) for p in predicted ]

# Hit if predicted technique is in the report's technique set
hits = [p in techniques_set for p in pred_upper]
correct = sum(hits)
total = len(pred_upper)

raw_overlap_acc = (correct / total) if total else 0.0
print(f"Raw overlap accuracy: {raw_overlap_acc*100:.2f}%  ({correct}/{total})")

# Inspecting / saving detailed results
overlap_df = pd.DataFrame({
    "Sentence": sentences,
    "PredictedTechnique": predicted,
    "Hit": hits
})
overlap_df.to_csv("cti_report_overlap_hits.csv", index=False)
print("Saved sentence-level hits to cti_report_overlap_hits.csv")


In [ ]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
precisions = []
recalls = []
corrected_pred = []
accepted_pred = []
correct_on_uniques = []
f1s = []

print(len(predicted))
print('list of predicted techniques: ',predicted)

lines = len(predicted)

In [ ]:
for threshold in thresholds:
    tecs = set(techniques)
    accepted = []

    for i in range(0,len(predict_proba_scores)):
        top_class = predicted[i]
        proba = predict_proba_scores[i]
        if proba > threshold:
            accepted.append(top_class)

    correct = 0

    unique_accepted = set(accepted)

    len_tecs = len(tecs)

    for pred in accepted:
        if pred in tecs: #True Positives
            correct += 1
    print('len of accepted: ',len(accepted))
    print('correct: ',correct)

    if len(accepted) != 0:
        precision = correct/len(accepted)*100
    else:
        precision = 0

    precision = round(precision,2)
    # print('precision: ',precision) #accuracy or precision?

    precisions.append(precision)

    for pred in accepted:
        if pred in tecs:
            tecs.remove(pred)

    recall = str(len_tecs-len(tecs))+ '/' + str(len_tecs)

    # print('recall: ',recall) #Recall

    recalls.append(recall)
    recall = (len_tecs-len(tecs))/len_tecs

    corrected_pred.append(correct)
    accepted_pred.append(len(accepted))

    cou = str(len_tecs-len(tecs))+ '/' + str(len(unique_accepted))
    correct_on_uniques.append(cou)
    cou = 0 if len(unique_accepted) == 0 else (len_tecs-len(tecs))/len(unique_accepted)

    # f1 = f_measure(recall=recall, precision=cou)
    f1 = f_measure(recall=recall, precision=precision / 100)
    f1 = round(f1,2)
    f1s.append(f1)

    print(f"Threshold {threshold:.2f}: F1 Score = {f1}, Precision = {round(precision,2)}, Recall = {round(recall,2)}")

In [ ]:
import matplotlib.pyplot as plt

# Converting string-form recall values to fractions
recalls_numeric = []
for r in recalls:
    if isinstance(r, str) and "/" in r:
        num, den = map(int, r.split("/"))
        recalls_numeric.append(num / den)
    else:
        recalls_numeric.append(r)

# Converting precision % to fraction (0-1 scale)
precisions_fraction = [p / 100 for p in precisions]

# Plot
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions_fraction, marker='o', label='Precision')
plt.plot(thresholds, recalls_numeric, marker='s', label='Recall')
plt.plot(thresholds, f1s, marker='^', label='F1 Score')

plt.xlabel('Confidence Threshold')
plt.ylabel('Score (0–1)')
plt.title('Precision, Recall, and F1 Score vs Threshold')
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
